# Antibody Data Harmonizer — Synthetic Demo

This notebook demonstrates a portfolio-safe workflow:

1. Load assay and Kabat annotation tables.
2. Standardize inconsistent column headers.
3. Normalize antibody IDs conservatively.
4. Merge with audit tables instead of silently dropping records.
5. Calculate CDR3 length.
6. Locate and mark a sequence feature.
7. Analyze CDR3 length against expression, HIC, and Tm.

All records in this notebook are synthetic.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Allow imports from the repository root when running this notebook.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import add_sequence_length, correlation_table
from src.feature_locator import locate_sequence_feature
from src.id_harmonization import (
    add_normalized_id,
    safe_merge,
    standardize_columns,
    suggest_id_matches,
)

pd.set_option("display.max_columns", 50)

## 1. Load synthetic sample data

In [ ]:
assay_path = PROJECT_ROOT / "data" / "sample" / "sample_assay_data.csv"
kabat_path = PROJECT_ROOT / "data" / "sample" / "sample_kabat_data.csv"

assay_raw = pd.read_csv(assay_path)
kabat_raw = pd.read_csv(kabat_path)

display(assay_raw.head())
display(kabat_raw.head())

## 2. Standardize headers

This converts headers such as `Expression Yield (mg/L)` into
`expression_yield_mg_l`, making downstream code easier to reuse.

In [ ]:
assay = standardize_columns(assay_raw)
kabat = standardize_columns(kabat_raw)

print("Assay columns:", assay.columns.tolist())
print("Kabat columns:", kabat.columns.tolist())

## 3. Normalize antibody IDs

Rules should be conservative and explicit. Here we remove punctuation and,
only for the assay table, explicitly allow the known synthetic suffix `HC`.

In [ ]:
assay = add_normalized_id(
    assay,
    source_column="antibody_code",
    removable_suffixes=("HC",),
)

kabat = add_normalized_id(
    kabat,
    source_column="protein_id",
)

display(assay[["antibody_code", "antibody_id"]])
display(kabat[["protein_id", "antibody_id"]])

## 4. Merge with an audit trail

An outer merge preserves unmatched records. `validate="one_to_one"` fails
loudly when duplicate IDs make the merge unsafe.

In [ ]:
merged, audit = safe_merge(
    assay,
    kabat,
    key="antibody_id",
    how="outer",
    validate="one_to_one",
)

print("Matched:", len(audit["matched"]))
print("Assay only:", len(audit["left_only"]))
print("Kabat only:", len(audit["right_only"]))
display(merged[["antibody_id", "antibody_code", "protein_id", "_merge"]])

## 5. Review unresolved IDs

Approximate matching creates review candidates only; it does not change IDs or
silently merge records.

In [ ]:
left_unresolved = audit["left_only"]["antibody_id"].dropna()
right_unresolved = audit["right_only"]["antibody_id"].dropna()

suggestions = suggest_id_matches(
    left_unresolved,
    right_unresolved,
    minimum_score=0.60,
)

display(suggestions if not suggestions.empty else "No candidate matches found.")

## 6. Calculate CDR3 length

In [ ]:
matched = audit["matched"].copy()
matched = add_sequence_length(
    matched,
    sequence_column="cdr3_sequence",
    output_column="cdr3_length",
)

display(
    matched[
        [
            "antibody_id",
            "cdr3_sequence",
            "cdr3_length",
            "expression_yield_mg_l",
            "hic_rt_min",
            "tm_c",
        ]
    ]
)

## 7. Locate and mark a sequence feature

The demo searches for the literal motif `WGQG`. The output includes presence,
count, one-based start positions, and a marked sequence.

In [ ]:
feature_hits = locate_sequence_feature(
    matched,
    id_column="antibody_id",
    sequence_column="heavy_sequence",
    feature="WGQG",
    feature_name="Example FR4 motif",
)

display(
    feature_hits[
        [
            "antibody_id",
            "feature_present",
            "feature_count",
            "start_positions_1based",
            "marked_sequence",
        ]
    ]
)

## 8. Correlation summary

In [ ]:
correlations = correlation_table(
    matched,
    predictor="cdr3_length",
    outcomes=["expression_yield_mg_l", "hic_rt_min", "tm_c"],
    methods=("spearman", "pearson"),
)

display(correlations)

## 9. Example visualization

The synthetic dataset is intentionally tiny, so this plot demonstrates the
workflow rather than supporting a biological conclusion.

In [ ]:
plot_data = matched[["cdr3_length", "expression_yield_mg_l"]].dropna()

plt.figure(figsize=(6, 4))
plt.scatter(plot_data["cdr3_length"], plot_data["expression_yield_mg_l"])
plt.xlabel("CDR3 length (aa)")
plt.ylabel("Expression yield (mg/L)")
plt.title("Synthetic demo: CDR3 length vs expression")
plt.tight_layout()
plt.show()

## 10. Export auditable outputs

In [ ]:
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

merged.to_csv(output_dir / "merged_audit.csv", index=False)
audit["matched"].to_csv(output_dir / "matched_records.csv", index=False)
audit["left_only"].to_csv(output_dir / "assay_only_records.csv", index=False)
audit["right_only"].to_csv(output_dir / "annotation_only_records.csv", index=False)
feature_hits.to_csv(output_dir / "feature_hits.csv", index=False)
correlations.to_csv(output_dir / "correlation_summary.csv", index=False)

print(f"Outputs written to: {output_dir}")